In [ ]:
import pandas as pd

processed_file_path = '../../data/processed/shopee_gold_dataset_absa_5_aspects.csv'
df = pd.read_csv(processed_file_path)

print("Kích thước dataset:", df.shape)
print("\nKiểm tra dữ liệu thiếu (null):\n", df.isnull().sum())

# Kiểm tra phân phối của các nhãn (Aspects)
aspects = ['label_quality', 'label_price', 'label_delivery']
for col in aspects:
    if col in df.columns:
        print(f"\n--- Phân phối của {col} ---")
        print(df[col].value_counts(dropna=False))

pd.set_option('display.max_colwidth', None)
display(df[['review_text'] + aspects].sample(5))

Kích thước dataset: (2999, 4)

Kiểm tra dữ liệu thiếu (null):
 review_text       0
label_quality     0
label_price       0
label_delivery    0
dtype: int64

--- Phân phối của label_quality ---
label_quality
negative    1257
positive    1186
none         357
neutral      199
Name: count, dtype: int64

--- Phân phối của label_price ---
label_price
none        2550
positive     335
negative     103
neutral       11
Name: count, dtype: int64

--- Phân phối của label_delivery ---
label_delivery
none        1394
negative     844
positive     727
neutral       34
Name: count, dtype: int64


,review_text,label_quality,label_price,label_delivery
1790,"kính trong sài rất oki nha mn , nên mua",positive,none,none
2701,Hương vị:sua thom ngot de uong Kiểu dáng:bai bi đẹp đóng gói cẩn thân Chất lượng:bo sung canxi dinh dưỡng,positive,none,positive
1355,Không được tặng 3 chai nhỏ ạ,negative,none,none
1692,"Giao hàng nhanh, đóng gói cẩn thận, bao bì mới",none,none,positive
2829,Sữa móp còn ráng giao nữa trong khi họ thanh toán rồi Khó chịu vô cùng nha,none,none,negative


In [4]:
from sklearn.model_selection import train_test_split

# Loại bỏ những dòng bị lỗi quá trình gán nhãn (nếu có NaN ở các cột quan trọng)
df_clean = df.dropna(subset=['review_text', 'label_quality', 'label_price', 'label_delivery', 'label_packaging', 'label_service'])

# Chia 80% cho tập train, 20% còn lại cho tập tạm (temp)
train_df, temp_df = train_test_split(df_clean, test_size=0.2, random_state=42)

# Chia đôi tập temp thành 10% validation và 10% test
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Số lượng mẫu - Train: {train_df.shape[0]} | Valid: {valid_df.shape[0]} | Test: {test_df.shape[0]}")

# Export ra các file CSV để dùng cho quá trình huấn luyện
train_df.to_csv('../../data/processed/train.csv', index=False)
valid_df.to_csv('../../data/processed/valid.csv', index=False)
test_df.to_csv('../../data/processed/test.csv', index=False)

print("Đã lưu thành công train.csv, valid.csv, và test.csv!")

Số lượng mẫu - Train: 2393 | Valid: 299 | Test: 300
Đã lưu thành công train.csv, valid.csv, và test.csv!


In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from underthesea import word_tokenize
import os
import gc

# CHỈ TRAIN ASPECT QUALITY THEO YÊU CẦU ĐỂ KHẮC PHỤC LỖI CATCH-ALL
NEW_ASPECTS = ['label_quality']

# Tùy chỉnh Trainer để ghi đè hàm tính Loss
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Sử dụng Weighted Cross-Entropy Loss
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


for ASPECT in NEW_ASPECTS:
    print(f"\n{'='*50}")
    print(f"🚀 Bắt đầu quá trình huấn luyện cho khía cạnh: {ASPECT}...")
    print(f"{'='*50}\n")

    # 1. ĐỌC DỮ LIỆU ĐÃ CHIA TỪ TRƯỚC
    train_df = pd.read_csv('../../data/processed/train.csv')
    valid_df = pd.read_csv('../../data/processed/valid.csv')

    # 2. CHUẨN HÓA NHÃN THÀNH SỐ (0, 1, 2, 3)
    label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2, 'none': 3}
    train_df['label'] = train_df[ASPECT].map(label_mapping)
    valid_df['label'] = valid_df[ASPECT].map(label_mapping)

    # Lọc bỏ dòng lỗi
    train_df = train_df.dropna(subset=['label', 'review_text'])
    valid_df = valid_df.dropna(subset=['label', 'review_text'])

    train_df['label'] = train_df['label'].astype(int)
    valid_df['label'] = valid_df['label'].astype(int)

    # TÍNH TOÁN CLASS WEIGHTS
    print("⚖️ Đang tính toán trọng số các nhãn (Class Weights)...")
    classes = np.unique(train_df['label'])
    balanced_weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_df['label'])
    weight_dict = dict(zip(classes, balanced_weights))
    
    # Áp dụng trọng số cho 4 class (mặc định 1.0 nếu thiếu)
    final_weights = [weight_dict.get(i, 1.0) for i in range(4)]
    
    # TĂNG TRỌNG SỐ CHO LỚP 'NONE' (nhãn 3) ĐỂ ÉP MÔ HÌNH HỌC KỸ HƠN
    # Hệ số x2.0 có thể tùy chỉnh nếu bạn muốn mô hình nhạy hơn nữa
    final_weights[3] = final_weights[3] * 2.0 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tensor_weights = torch.tensor(final_weights, dtype=torch.float).to(device)
    print(f"   Trọng số áp dụng (0: Neg, 1: Neu, 2: Pos, 3: None): {np.round(final_weights, 3)}")

    # 3. TÁCH TỪ TIẾNG VIỆT (WORD SEGMENTATION)
    print("📝 Đang tiến hành tách từ tiếng Việt (Word Segmentation)...")
    def segment_text(text):
        return word_tokenize(str(text), format="text")

    train_df['review_segmented'] = train_df['review_text'].apply(segment_text)
    valid_df['review_segmented'] = valid_df['review_text'].apply(segment_text)

    # 4. KHỞI TẠO TOKENIZER 
    model_name = "vinai/phobert-base-v2"  
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(examples):
        return tokenizer(examples['review_segmented'], padding="max_length", truncation=True, max_length=128)

    train_dataset = Dataset.from_pandas(train_df[['review_segmented', 'label']])
    valid_dataset = Dataset.from_pandas(valid_df[['review_segmented', 'label']])

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    valid_dataset = valid_dataset.map(tokenize_function, batched=True)

    # 5. ĐỊNH NGHĨA HÀM ĐÁNH GIÁ TRONG QUÁ TRÌNH TRAIN
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = logits.argmax(axis=-1)
        
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
        acc = accuracy_score(labels, predictions)
        
        return {
            'accuracy': acc,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }

    # 6. KHỞI TẠO MÔ HÌNH PHOBERT
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

    # 7. THIẾT LẬP THAM SỐ HUẤN LUYỆN
    training_args = TrainingArguments(
        output_dir=f"./results_{ASPECT}",
        learning_rate=2e-5,                  
        per_device_train_batch_size=16,      
        per_device_eval_batch_size=16,
        num_train_epochs=3,                  
        weight_decay=0.01,
        eval_strategy="epoch",         
        save_strategy="epoch",
        load_best_model_at_end=True,         
        metric_for_best_model="f1",
        logging_steps=50,
        fp16=torch.cuda.is_available(),     
        report_to="none"
    )

    # 8. KHỞI TẠO TRAINER (Sử dụng WeightedTrainer tùy chỉnh)
    trainer = WeightedTrainer(
        class_weights=tensor_weights,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        compute_metrics=compute_metrics,
    )

    # 9. TIẾN HÀNH TRAINING
    print(f"🏋️‍♂️ Đang huấn luyện mô hình {ASPECT}...")
    trainer.train()

    # 10. ĐÁNH GIÁ (CLASSIFICATION REPORT & CONFUSION MATRIX) TRÊN TẬP VALIDATION
    print("\n" + "-"*50)
    print("📊 ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP VALIDATION")
    print("-"*50)
    
    predictions_output = trainer.predict(valid_dataset)
    preds = predictions_output.predictions.argmax(-1)
    labels = predictions_output.label_ids
    
    target_names = ['Negative (0)', 'Neutral (1)', 'Positive (2)', 'None (3)']
    
    print("\nClassification Report:")
    print(classification_report(labels, preds, labels=[0, 1, 2, 3], target_names=target_names, zero_division=0))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(labels, preds, labels=[0, 1, 2, 3]))
    print("-" * 50 + "\n")

    # 11. XUẤT VÀ LƯU MÔ HÌNH
    output_model_path = f"../../models/{ASPECT}_model_weighted"
    os.makedirs(output_model_path, exist_ok=True)
    model.save_pretrained(output_model_path)
    tokenizer.save_pretrained(output_model_path)

    print(f"🎉 Hoàn thành! Mô hình đã được lưu thành công tại: {output_model_path}")
    
    # 12. GIẢI PHÓNG BỘ NHỚ
    del model, trainer, train_dataset, valid_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
print("\n" + "="*50)
print("✅ ĐÃ HOÀN TẤT TRAIN MÔ HÌNH QUALITY VỚI WEIGHTED CROSS-ENTROPY LOSS.")
print("="*50)

d:\doan3_phantichcamxuckhachhang\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🚀 Bắt đầu quá trình huấn luyện cho khía cạnh: label_quality...

⚖️ Đang tính toán trọng số các nhãn (Class Weights)...
   Trọng số áp dụng (0: Neg, 1: Neu, 2: Pos, 3: None): [0.843 2.835 0.666 2.084]
📝 Đang tiến hành tách từ tiếng Việt (Word Segmentation)...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 26388.54it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🏋️‍♂️ Đang huấn luyện mô hình label_quality...


d:\doan3_phantichcamxuckhachhang\env\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...


--------------------------------------------------
📊 ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP VALIDATION
--------------------------------------------------


d:\doan3_phantichcamxuckhachhang\env\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Classification Report:
              precision    recall  f1-score   support

Negative (0)       0.77      0.69      0.73        77
 Neutral (1)       0.33      0.39      0.36        23
Positive (2)       0.88      0.86      0.87       125
    None (3)       0.71      0.77      0.74        74

    accuracy                           0.76       299
   macro avg       0.67      0.68      0.67       299
weighted avg       0.77      0.76      0.76       299


Confusion Matrix:
[[ 53   3   4  17]
 [  4   9   8   2]
 [  1  12 108   4]
 [ 11   3   3  57]]
--------------------------------------------------



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.20it/s]


🎉 Hoàn thành! Mô hình đã được lưu thành công tại: ../../models/label_quality_model_weighted

✅ ĐÃ HOÀN TẤT TRAIN MÔ HÌNH QUALITY VỚI WEIGHTED CROSS-ENTROPY LOSS.


In [10]:
import json
import pandas as pd
import os

def get_training_history(aspect_name):
    # Tìm thư mục results của aspect
    result_dir = f"./results_{aspect_name}"
    
    if not os.path.exists(result_dir):
        print(f"❌ Không tìm thấy thư mục kết quả: {result_dir}")
        return None
        
    # Tìm checkpoint cuối cùng hoặc file state bên trong
    checkpoints = [d for d in os.listdir(result_dir) if d.startswith("checkpoint")]
    if not checkpoints:
        print(f"❌ Không tìm thấy checkpoint nào trong {result_dir}")
        return None
        
    # Sắp xếp để lấy checkpoint mới nhất/cao nhất
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    latest_checkpoint = checkpoints[-1]
    state_file_path = os.path.join(result_dir, latest_checkpoint, "trainer_state.json")
    
    if not os.path.exists(state_file_path):
        print(f"❌ Không tìm thấy file trainer_state.json tại {state_file_path}")
        return None
        
    # Đọc file json lịch sử
    with open(state_file_path, "r") as f:
        state_data = json.load(f)
        
    # Lọc ra các dòng ghi nhận kết quả đánh giá (evaluation) sau mỗi epoch
    history = state_data.get("log_history", [])
    eval_logs = [log for log in history if "eval_loss" in log]
    
    # Chuyển thành DataFrame cho dễ nhìn
    df_metrics = pd.DataFrame(eval_logs)
    
    # Chọn và đổi tên các cột cần thiết giống bảng hiển thị khi train
    columns_to_keep = {
        'epoch': 'Epoch',
        'eval_loss': 'Validation Loss',
        'eval_accuracy': 'Accuracy',
        'eval_f1': 'F1',
        'eval_precision': 'Precision',
        'eval_recall': 'Recall'
    }
    
    # Kiểm tra xem có đủ cột không (phòng trường hợp log thiếu)
    existing_cols = {k: v for k, v in columns_to_keep.items() if k in df_metrics.columns}
    df_metrics = df_metrics[list(existing_cols.keys())].rename(columns=existing_cols)
    
    print(f"\n📊 BẢNG CHỈ SỐ LỊCH SỬ CỦA: {aspect_name.upper()}")
    return df_metrics

# --- CHẠY ĐỂ XEM KẾT QUẢ ---
# Xem mô hình Price
df_price = get_training_history('label_price')
if df_price is not None:
    display(df_price)

# Xem mô hình Delivery
df_delivery = get_training_history('label_delivery')
if df_delivery is not None:
    display(df_delivery)


📊 BẢNG CHỈ SỐ LỊCH SỬ CỦA: LABEL_PRICE


,Epoch,Validation Loss,Accuracy,F1,Precision,Recall
0,1.0,0.219017,0.946667,0.603895,0.568972,0.649653
1,2.0,0.173670,0.943333,0.588079,0.544152,0.659004
2,3.0,0.156530,0.943333,0.721204,0.712791,0.732597



📊 BẢNG CHỈ SỐ LỊCH SỬ CỦA: LABEL_DELIVERY


,Epoch,Validation Loss,Accuracy,F1,Precision,Recall
0,1.0,0.441301,0.880000,0.663241,0.664923,0.662577
1,2.0,0.413429,0.863333,0.651387,0.640843,0.665935
2,3.0,0.407148,0.873333,0.658324,0.648904,0.670065


In [ ]:
import pickle

# Đường dẫn trỏ tới file pkl
with open('../../models/item_id_map.pkl', 'rb') as f:
    user_id_map = pickle.load(f)

# Kiểm tra kiểu dữ liệu
print("Kiểu dữ liệu:", type(user_id_map))

# Nếu nó là Dictionary, in thử 10 User ID đầu tiên để test
if isinstance(user_id_map, dict):
    print("--- 10 User ID đầu tiên ---")
    for i, (key, value) in enumerate(user_id_map.items()):
        print(f"Key: {key} -> Mapped Value: {value}")
        if i >= 9: break
else:
    # Nếu là dạng khác (list, dataframe), in ra 5 dòng đầu
    print(user_id_map[:5])

Kiểu dữ liệu: <class 'dict'>
--- 10 User ID đầu tiên ---
Key: 0 -> Mapped Value: 159901
Key: 1 -> Mapped Value: 476608
Key: 2 -> Mapped Value: 481290
Key: 3 -> Mapped Value: 497990
Key: 4 -> Mapped Value: 521639
Key: 5 -> Mapped Value: 564603
Key: 6 -> Mapped Value: 564604
Key: 7 -> Mapped Value: 565575
Key: 8 -> Mapped Value: 565584
Key: 9 -> Mapped Value: 570489


In [3]:
import pickle

# Lưu ý: Sửa lại đường dẫn cho đúng vì EDA2.ipynb đang nằm trong thư mục con
# Cần lùi ra ngoài 2 cấp (../../) để vào thư mục models
with open('../../models/user_id_map.pkl', 'rb') as f:
    user_mapping = pickle.load(f)

# Xem cấu trúc thật sự
items = list(user_mapping.items())[:5]
print("5 phần tử đầu tiên:", items)
print("Kiểu key:", type(items[0][0]), "| Kiểu value:", type(items[0][1]))

5 phần tử đầu tiên: [(0, 'USR_0001'), (1, 'USR_0002'), (2, 'USR_0003'), (3, 'USR_0004'), (4, 'USR_0005')]
Kiểu key: <class 'int'> | Kiểu value: <class 'str'>


In [ ]:
import pandas as pd

# 1. Tải 2 bảng dữ liệu
try:
    df_review = pd.read_csv('../../data/processed/review_analysis_full.csv')
    df_product = pd.read_csv('../../data/processed/products.csv')
    
    # 2. Tìm tên cột ID trong bảng Review
    foreign_key = next((col for col in ['product_id', 'item_id', 'id'] if col in df_review.columns), None)
    
    if not foreign_key:
        print("❌ LỖI: Bảng Review không có cột ID sản phẩm nào (product_id, item_id, id).")
    else:
        print(f"✅ Đã tìm thấy cột liên kết trong bảng Review: '{foreign_key}'")
        
        # 3. Trích xuất danh sách ID duy nhất
        review_ids = set(df_review[foreign_key].dropna().unique())
        product_ids = set(df_product['id'].dropna().unique())
        common_ids = review_ids.intersection(product_ids)
        
        print("\n📊 BÁO CÁO THỐNG KÊ ID:")
        print(f"- Số lượng ID duy nhất trong Review: {len(review_ids)}")
        print(f"- Số lượng ID duy nhất trong Product: {len(product_ids)}")
        print(f"- Số lượng ID TRÙNG KHỚP: {len(common_ids)}")
        
        print("\n🕵️ MẪU DỮ LIỆU THỰC TẾ (Nhìn vào đây để thấy tại sao lệch):")
        print(f"👉 5 ID đầu tiên trong Review : {list(review_ids)[:5]}")
        print(f"👉 5 ID đầu tiên trong Product: {list(product_ids)[:5]}")
        
        # 4. Phân tích kiểu dữ liệu
        print("\n🧬 KIỂU DỮ LIỆU (DATA TYPE):")
        print(f"- Kiểu của ID trong Review : {type(list(review_ids)[0])}")
        print(f"- Kiểu của ID trong Product: {type(list(product_ids)[0])}")

except Exception as e:
    print(f"❌ Lỗi khi đọc file: {e}")

🔍 ĐANG QUÉT DỮ LIỆU ĐỂ TÌM NGUYÊN NHÂN LỆCH ID...
✅ Đã tìm thấy cột liên kết trong bảng Review: 'id'

📊 BÁO CÁO THỐNG KÊ ID:
- Số lượng ID duy nhất trong Review: 2999
- Số lượng ID duy nhất trong Product: 41576
- Số lượng ID TRÙNG KHỚP: 0

🕵️ MẪU DỮ LIỆU THỰC TẾ (Nhìn vào đây để thấy tại sao lệch):
👉 5 ID đầu tiên trong Review : [np.int64(48016973830), np.int64(52431331336), np.int64(80611655688), np.int64(18397110288), np.int64(84932157457)]
👉 5 ID đầu tiên trong Product: [np.int64(102629376), np.int64(54001669), np.int64(177602569), np.int64(89653263), np.int64(114688019)]

🧬 KIỂU DỮ LIỆU (DATA TYPE):
- Kiểu của ID trong Review : <class 'numpy.int64'>
- Kiểu của ID trong Product: <class 'numpy.int64'>


In [5]:
import pandas as pd

df = pd.read_csv(
    '../../data/processed/shopee_gold_dataset_absa_5_aspects.csv'
)

print("Kích thước:", df.shape)
print("\nCác cột:")
print(df.columns.tolist())

print("\n5 dòng đầu:")
display(df.head())

Kích thước: (2992, 6)

Các cột:
['review_text', 'label_quality', 'label_price', 'label_delivery', 'label_packaging', 'label_service']

5 dòng đầu:


,review_text,label_quality,label_price,label_delivery,label_packaging,label_service
0,"Shop giao hàng nhanh, đúng số lượng và mô tả, ...",positive,none,positive,negative,none
1,"Hơi thất vọng vì ko còn là mẫu cũ, dạng sữa, g...",negative,none,none,none,none
2,Bao bì/Mẫu mã:đúng Hương vị:mix vị Hàng giao ...,negative,none,positive,negative,negative
3,"Hình ảnh mang tính chất nhận xu, sản phẩm y hì...",positive,none,none,positive,none
4,Lợi ích:sạch da Làm đẹp:ổn Kinh nghiệm sử dụng...,positive,none,none,negative,none
